In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

In [ ]:
!pip install aisuite[all] datasets openai rouge-score

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")
openai_token = user_secrets.get_secret("openai")

In [ ]:
os.environ['OPENAI_API_KEY'] = openai_token
# os.environ['ANTHROPIC_API_KEY'] = 'your-anthropic-api-key'
os.environ["HUGGINGFACE_TOKEN"] = hf_token

In [ ]:
!huggingface-cli login --token $hf_token --add-to-git-credential

In [ ]:
# !pip install aisuite[huggingface]

In [ ]:
import aisuite as ai
client = ai.Client()

In [ ]:
# models = ["huggingface:mistralai/Mistral-7B-Instruct-v0.3"]

# messages = [
#     {"role": "system", "content": "Respond in Pirate English. Always try to include the phrase - No rum No fun."},
#     {"role": "user", "content": "Tell me a joke about Captain Jack Sparrow"},
# ]

# for model in models:
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.75,
#         max_tokens=512,

#     )
#     print(response.choices[0].message.content)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0", split='test')
sample_articles = dataset.select(range(50))  # Limit to 1 article for demonstration

In [ ]:
sample_articles

#### Zero Shot Prompting

In [ ]:
def zero_shot_prompt(article, model):
    messages = [
        {"role": "user", "content": f"Summarize the following article:\n\n{article}\n\nFinal Summary:"}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

#### Few Shot Prompting

In [ ]:
def generate_few_shot_examples(dataset, num_examples=2):
    examples = ""
    for i in range(num_examples):
        article = dataset[i]["article"]
        summary = dataset[i]["highlights"]
        examples += f"Article: {article}\nSummary: {summary}\n\n"
    return examples

few_shot_examples = generate_few_shot_examples(sample_articles)

def few_shot_prompt(article, model, few_shot_examples):
    messages = [
        {"role": "user", "content": few_shot_examples + f"Article: {article}\n\nFinal Summary:"}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

#### CoT

In [ ]:
def cot_prompt(article, model):
    messages = [
        {"role": "user", "content": (
            f"Summarize the following article step by step:\n\n{article}\n\n"
            "Step 1: Identify the key points of the article.\n"
            "Step 2: Combine these key points into a concise summary.\n\n"
            "Final Summary:"
        )}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

#### Self Consistency

In [ ]:
def self_consistency_prompt(article, model):
    summaries = []
    for _ in range(5):  # Generate 5 summaries
        messages = [
            {"role": "user", "content": f"Summarize:\n{article}"}
        ]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.7
        )
        summaries.append(response.choices[0].message.content)
    return max(set(summaries), key=summaries.count)  # Choose the most frequent summary

#### CoVe

In [ ]:
def cove_prompt(article, model):
    messages = [
        {"role": "user", "content": (
            f"Summarize the article. After summarizing, verify each sentence against the text:\n\n{article}\n\n"
            "Final Summary:"
        )}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

#### Active Prompt

In [ ]:
# def active_prompt(article, model):
#     initial_messages = [
#         {"role": "user", "content": f"Summarize the article:\n\n{article}"}
#     ]
#     summary = client.chat.completions.create(
#         model=model,
#         messages=initial_messages,
#         temperature=0.7
#     ).choices[0].message.content

#     user_feedback = input(f"Final Summary:\n{summary}\nModify? (Yes/No): ")
#     if user_feedback.lower() == "yes":
#         refined_messages = [
#             {"role": "user", "content": f"The user suggested changes to this summary:\n\n{summary}\n\nNew Summary:"}
#         ]
#         return client.chat.completions.create(
#             model=model,
#             messages=refined_messages,
#             temperature=0.7
#         ).choices[0].message.content
#     return summary

#### SCoT

In [ ]:
def scot_prompt(article, model):
    messages = [
        {"role": "user", "content": (
            f"Summarize the article using the following structure:\n"
            "1. Background\n"
            "2. Key Points\n"
            "3. Conclusion\n\n"
            f"{article}"
            "Final Summary:"
        )}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content


#### Logical CoT

In [ ]:
# def logical_cot_prompt(article, model):
#     messages = [
#         {"role": "user", "content": (
#             f"Summarize the following article logically:\n\n{article}\n\n"
#             "Step 1: What is the main topic of the article?\n"
#             "Step 2: List three key points supporting the topic.\n"
#             "Step 3: Summarize each key point.\n"
#             "Step 4: Combine these summaries into one concise summary. Final Summary:"
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

#### Tree-of-Thought (ToT)

In [ ]:
# def tot_prompt(article, model):
#     prompt = (
#         f"Summarize the following article by generating multiple summaries:\n\n{article}\n\n"
#         "Option 1: Summarize focusing on key points.\n"
#         "Option 2: Summarize focusing on the emotional tone of the article.\n"
#         "Option 3: Summarize focusing on causal relationships in the article.\n\n"
#         "Choose the best summary from the options above.\n\n"
#         "Final Summary:"
#     )
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

#### Graph-of-Thought (GoT)

In [ ]:
# def graph_of_thought_prompt(article, model):
#     messages = [
#         {"role": "user", "content": (
#             f"Summarize the following article by creating a graph of relationships:\n\n{article}\n\n"
#             "Structure the summary as:\n"
#             "Main Idea -> Supporting Point 1 -> Sub-point 1.1 -> Sub-point 1.2 -> Supporting Point 2...\n\n"
#             "Final Summary:"
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

#### Chain-of-Symbols (CoS)

In [ ]:
# def cos_prompt(article, model):
#     messages = [
#         {"role": "user", "content": (
#             f"Summarize the article in the form of symbols or a bulleted list:\n\n{article}\n\n"
#             "- Main Topic:\n"
#             "- Key Point 1:\n"
#             "- Key Point 2:\n"
#             "- Conclusion:\n\n"
#             "Final Summary:"
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content


#### Rephrase and Respond (RaR)

In [ ]:
# def generate_initial_summary(article, model):
#     messages = [
#         {"role": "user", "content": f"Summarize the following article:\n\n{article}"}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

# def cross_check_factuality(article, summary, model):
#     messages = [
#         {"role": "user", "content": (
#             f"The following summary has been generated:\n\n{summary}\n\n"
#             f"Cross-check this summary against the original article below and ensure it is factually accurate."
#             f"If any parts are incorrect, rewrite them for factual correctness:\n\n{article}\n\n"
#             f"Final Summary:"
            
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

# def rar_prompt(article, initial_summary, model, iterations=2):
#     refined_summary = initial_summary
#     for _ in range(iterations):
#         refined_summary = cross_check_factuality(article, refined_summary, model)
#     return refined_summary

#### Take a Step Back

In [ ]:
# def step_back_prompt(article, model):
#     messages = [
#         {"role": "user", "content": (
#             f"Take a step back and summarize the article by focusing on its broader context:\n\n{article}\n\n"
#             "What are the key takeaways from this article in a broader perspective?\n\n"
#             "Final Summary:"
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

#### ScratchPad

In [ ]:
# def scratchpad_prompt(article, model):
#     messages = [
#         {"role": "user", "content": (
#             f"Use a scratchpad to organize your thoughts before summarizing the article:\n\n{article}\n\n"
#             "Scratchpad:\n- Main Topic:\n- Key Details:\n- Supporting Points:\n\n"
#             "Final Summary:"
#         )}
#     ]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.7
#     )
#     return response.choices[0].message.content

### Batch Processing Pipeline

In [ ]:
def batch_process_strategies(dataset, models, strategies, max_articles=10):
    results = {}

    for i in range(min(max_articles, len(dataset))):
        article = dataset[i]["article"]
        reference_summary = dataset[i]["highlights"]
        
        results[i] = {"article": article, "reference": reference_summary, "strategies": {}}
        
        for model in models:
            results[i]["strategies"][model] = {}
            
            for strategy_name, strategy_function in strategies.items():
                try:
                    generated_summary = strategy_function(article, model)
                    results[i]["strategies"][model][strategy_name] = generated_summary
                except Exception as e:
                    results[i]["strategies"][model][strategy_name] = f"Error: {e}"
    
    return results

In [ ]:
# List of models to use
models = [
    # "openai:gpt-4o",
    "openai:gpt-4o-mini", 
    "openai:gpt-3.5-turbo-0125",

    # "huggingface:google-t5/t5-base",
    # "huggingface:google/flan-t5-large",
    # "huggingface:google/pegasus-large",
    
    # "huggingface:facebook/bart-large-cnn",
    "huggingface:mistralai/Mistral-7B-Instruct-v0.3",
    
    # "huggingface:DISLab/SummLlama3.2-3B",
    # "huggingface:DISLab/SummLlama3.1-8B",
    # "huggingface:DISLab/SummLlama3-8B",
    # "huggingface:meta-llama/Llama-2-7b-chat-hf"
    

    # "huggingface:lmsys/vicuna-13b-v1.5",
    # "huggingface:microsoft/Orca-2-13b",
    "huggingface:microsoft/Phi-3.5-mini-instruct"
    
    
]

In [ ]:
# Dictionary of summarization strategies
strategies = {
    "Zero": zero_shot_prompt,
    "Few": lambda article, model: few_shot_prompt(article, model, few_shot_examples),
    "CoT": cot_prompt,
    "Self": self_consistency_prompt,
    "CoVe": cove_prompt,
    # "Active": active_prompt,
    "SCoT": scot_prompt,
    # "Logical CoT": logical_cot_prompt,
    # "Graph of Thought": graph_of_thought_prompt,
    # "CoS": cos_prompt,
    # "RaR": lambda article, model: rar_prompt(article, generate_initial_summary(article, model), model),
    # "StepBack": step_back_prompt,
    # "Scratchpad": scratchpad_prompt
}

# Run batch processing to generate summaries
results = batch_process_strategies(sample_articles, models, strategies)

In [ ]:
output_data = []

for article_id, result in results.items():
    row = {
        "article_id": article_id,
        "article": result["article"],
        "highlights": result["reference"]
    }
    for model, strategies in result["strategies"].items():
        for strategy, summary in strategies.items():
            model_name = model.split(":")[-1].replace("/", "_").replace(".", "").replace("-", "")
            column_name = f"{model_name}_{strategy.replace(' ', '')}"
            row[column_name] = summary.strip()
    output_data.append(row)

# Convert the data to a pandas DataFrame
df = pd.DataFrame(output_data)

output_csv_path = "summarization_results_combined.csv"
df.to_csv(output_csv_path, index=False, encoding='utf-8')